# Gold_Build_LAH_DF2

Agrega Silver BIX + Silver OneSite en tablas Gold para el Semantic Model.

| Tabla Gold | Fuente Silver |
|---|---|
| `gold_property_summary` | dimproperty + factpropertystatisticsasofdate + factoperationalkpi |
| `gold_ar_summary` | os_ar_collections |
| `gold_maintenance_summary` | dimservicerequest (WorkingDays, MRWorkingDays) |
| `gold_leasing_summary` | os_move_ins + os_denies_cancells |
| `gold_financial_summary` | factglsummary + dimaccountgrouphierarchy |

## Config

In [ ]:
# ============================================================
# Gold_Build_LAH_DF2 — CONFIG
# ============================================================
# Attach this notebook to Heritage_Gold_LH (default lakehouse)
# Also add Heritage_Silver_LH as a second (read-only) lakehouse
# ============================================================

SILVER_BIX_DB = "Heritage_Silver_LH"    # Silver BIX lakehouse DB name in Fabric
SILVER_OS_DB  = "Heritage_Silver_LH"    # Same lakehouse; OneSite tables prefixed os_

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

print(f"Silver DB  : {SILVER_BIX_DB}")
print(f"Gold DB    : (current default lakehouse)")

## Helpers

In [ ]:
# ============================================================
# HELPERS
# ============================================================
def _tbl(name):
    # Try Silver LH table, return None if not available
    try:
        return spark.table(name)
    except Exception as e:
        print(f"  WARN: could not load {name}: {e}")
        return None

def _save(df, tbl):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(tbl)
    cnt = spark.table(tbl).count()
    print(f"  Saved: {tbl} | rows={cnt:,}")
    return cnt

print("Helpers loaded.")

## Gold 1 — Property Summary

In [ ]:
# ============================================================
# GOLD 1 — gold_property_summary
# Property-level KPIs: units, occupancy, rent
# ============================================================
def build_gold_property_summary():
    props  = _tbl("dimproperty__0001")
    opstats = _tbl("factpropertystatisticsasofdate__0001")
    kpi    = _tbl("factoperationalkpi__0001")

    if props is None:
        print("  SKIP: dimproperty__0001 not available")
        return

    base = props.filter(F.col("IsCurrent") == 1).select(
        "PropertyKey",
        F.col("PropertyName").alias("Property"),
        F.col("PropertyAddress1").alias("Address"),
        F.col("OrganizationKey"),
    )

    if opstats is not None:
        latest = opstats.filter(F.col("IsCurrent") == 1)
        numeric_cols = ["TotalUnits","VacantUnits","OccupiedUnits",
                        "AverageMarketRent","AverageEffectiveRent",
                        "AverageActualRent","NtmOccupancyRate"]
        for c in numeric_cols:
            if c in latest.columns:
                latest = latest.withColumn(c, F.col(c).cast(DoubleType()))
        base = base.join(latest.select("PropertyKey", *[c for c in numeric_cols if c in latest.columns]),
                         on="PropertyKey", how="left")

    if kpi is not None and "PropertyKey" in kpi.columns:
        kpi_curr = kpi.filter(F.col("IsCurrent") == 1)
        kpi_cols = [c for c in kpi_curr.columns
                    if c not in ("PropertyKey","hash_value","Start_Date","End_Date","IsCurrent")]
        base = base.join(kpi_curr.select("PropertyKey", *kpi_cols[:10]), on="PropertyKey", how="left")

    base = base.withColumn("_gold_load_date", F.current_date())
    _save(base, "gold_property_summary")
    base.select("Property","TotalUnits","OccupiedUnits","AverageEffectiveRent").show(10, truncate=False)

## Gold 2 — AR Summary

In [ ]:
# ============================================================
# GOLD 2 — gold_ar_summary
# AR aging by property (from OneSite os_ar_collections)
# ============================================================
def build_gold_ar_summary():
    ar = _tbl("os_ar_collections")
    if ar is None:
        print("  SKIP: os_ar_collections not available")
        return

    for c in ["Current","Days_30_59","Days_60_89","Days_90_plus","Total"]:
        if c in ar.columns:
            ar = ar.withColumn(c, F.col(c).cast(DoubleType()))

    ar = ar.withColumn("Days_30_plus",
        F.coalesce(F.col("Days_30_59"),F.lit(0.0)) +
        F.coalesce(F.col("Days_60_89"),F.lit(0.0)) +
        F.coalesce(F.col("Days_90_plus"),F.lit(0.0))
    ).withColumn("Pct_Current",
        F.when(F.col("Total") != 0, F.col("Current") / F.col("Total") * 100).otherwise(F.lit(None).cast(DoubleType()))
    ).withColumn("Pct_Delinquent",
        F.when(F.col("Total") != 0, F.col("Days_30_plus") / F.col("Total") * 100).otherwise(F.lit(None).cast(DoubleType()))
    ).withColumn("_gold_load_date", F.current_date())

    _save(ar, "gold_ar_summary")
    ar.select("Property","Current","Days_30_59","Days_60_89","Days_90_plus","Total",
              "Pct_Delinquent").show(10, truncate=False)

## Gold 3 — Maintenance Summary

In [ ]:
# ============================================================
# GOLD 3 — gold_maintenance_summary
# Service requests: open, completed, avg working days
# ============================================================
def build_gold_maintenance_summary():
    sr = _tbl("dimservicerequest__0001")
    props = _tbl("dimproperty__0001")
    if sr is None:
        print("  SKIP: dimservicerequest__0001 not available")
        return

    sr_curr = sr.filter(F.col("IsCurrent") == 1)

    for c in ["WorkingDays","MRWorkingDays"]:
        if c in sr_curr.columns:
            sr_curr = sr_curr.withColumn(c, F.col(c).cast(DoubleType()))

    status_col = next((c for c in sr_curr.columns if "status" in c.lower()), None)
    type_col   = next((c for c in sr_curr.columns if "typecode" in c.lower() or "type" in c.lower()), None)

    agg_exprs = [
        F.count("*").alias("Total_SRs"),
        F.round(F.avg("WorkingDays"),1).alias("Avg_WorkingDays"),
        F.round(F.avg("MRWorkingDays"),1).alias("Avg_MR_WorkingDays"),
        F.sum(F.when(F.col("WorkingDays") > 3, 1).otherwise(0)).alias("SRs_Over_3Days"),
    ]
    if status_col:
        agg_exprs.append(F.sum(F.when(F.lower(F.col(status_col)).contains("open"),1).otherwise(0)).alias("Open_SRs"))
        agg_exprs.append(F.sum(F.when(F.lower(F.col(status_col)).contains("complet"),1).otherwise(0)).alias("Completed_SRs"))
    if type_col:
        agg_exprs.append(F.sum(F.when(F.upper(F.col(type_col))=="MR",1).otherwise(0)).alias("MakeReady_SRs"))

    grp_cols = ["PropertyKey"]
    if type_col: grp_cols.append(type_col)
    summary = sr_curr.groupBy(*grp_cols).agg(*agg_exprs)

    if props is not None:
        pnames = props.filter(F.col("IsCurrent")==1).select("PropertyKey", F.col("PropertyName").alias("Property"))
        summary = summary.join(pnames, on="PropertyKey", how="left")

    summary = summary.withColumn("_gold_load_date", F.current_date())
    _save(summary, "gold_maintenance_summary")
    summary.select("Property","Total_SRs","Open_SRs","Avg_WorkingDays").show(10, truncate=False)

## Gold 4 — Leasing Summary

In [ ]:
# ============================================================
# GOLD 4 — gold_leasing_summary
# Move-ins, denies, effective rents by property
# ============================================================
def build_gold_leasing_summary():
    movein  = _tbl("os_move_ins")
    denies  = _tbl("os_denies_cancells")
    eff_r   = _tbl("os_effective_rents")

    parts = []

    if movein is not None:
        prop_col = next((c for c in movein.columns if "property" in c.lower()), None)
        if prop_col:
            mi = movein.groupBy(F.col(prop_col).alias("Property")).agg(
                F.count("*").alias("MoveIns")).withColumn("_src","move_ins")
            parts.append(mi)

    if denies is not None:
        prop_col = next((c for c in denies.columns if "property" in c.lower()), None)
        if prop_col:
            dn = denies.groupBy(F.col(prop_col).alias("Property")).agg(
                F.count("*").alias("Denies")).withColumn("_src","denies")
            parts.append(dn)

    if not parts:
        print("  SKIP: no leasing data available")
        return

    from functools import reduce
    from pyspark.sql import DataFrame
    df = reduce(lambda a, b: a.join(b.drop("_src"), on="Property", how="outer"), parts)
    df = df.withColumn("_gold_load_date", F.current_date()).drop("_src")
    _save(df, "gold_leasing_summary")
    df.show(10, truncate=False)

## Gold 5 — Financial Summary

In [ ]:
# ============================================================
# GOLD 5 — gold_financial_summary
# GL summary by property and account group
# ============================================================
def build_gold_financial_summary():
    gl     = _tbl("factglsummary__0001")
    accts  = _tbl("dimaccountgrouphierarchy__0001")
    props  = _tbl("dimproperty__0001")
    if gl is None:
        print("  SKIP: factglsummary__0001 not available")
        return

    gl_curr = gl.filter(F.col("IsCurrent") == 1)

    amount_col = next((c for c in gl_curr.columns if "amount" in c.lower() or "balance" in c.lower()), None)
    period_col = next((c for c in gl_curr.columns if "period" in c.lower() or "postdate" in c.lower() or "month" in c.lower()), None)
    acct_col   = next((c for c in gl_curr.columns if "accountgrouphierarchykey" in c.lower()), None)

    grp_cols = ["PropertyKey"]
    if period_col: grp_cols.append(period_col)
    if acct_col:   grp_cols.append(acct_col)

    agg_exprs = [F.count("*").alias("GL_Rows")]
    if amount_col:
        gl_curr = gl_curr.withColumn(amount_col, F.col(amount_col).cast(DoubleType()))
        agg_exprs += [
            F.sum(amount_col).alias("Total_Amount"),
            F.round(F.avg(amount_col), 2).alias("Avg_Amount"),
        ]

    summary = gl_curr.groupBy(*grp_cols).agg(*agg_exprs)

    if props is not None:
        pnames = props.filter(F.col("IsCurrent")==1).select("PropertyKey", F.col("PropertyName").alias("Property"))
        summary = summary.join(pnames, on="PropertyKey", how="left")

    summary = summary.withColumn("_gold_load_date", F.current_date())
    _save(summary, "gold_financial_summary")
    summary.show(10, truncate=False)

## Gold 6 — Validation

In [ ]:
# ============================================================
# GOLD 6 — VALIDATION
# ============================================================
GOLD_TABLES = [
    "gold_property_summary",
    "gold_ar_summary",
    "gold_maintenance_summary",
    "gold_leasing_summary",
    "gold_financial_summary",
]

def validate_gold():
    checks = []
    print("=== Gold Layer Row Counts ===")
    for tbl in GOLD_TABLES:
        try:
            cnt = spark.table(tbl).count()
            ok  = cnt > 0
            checks.append({"tbl": tbl, "cnt": cnt, "ok": ok})
            print(f"  {'OK' if ok else 'WARN'} {tbl}: {cnt:,} rows")
        except Exception as e:
            checks.append({"tbl": tbl, "cnt": -1, "ok": False})
            print(f"  WARN {tbl}: {e}")

    print("\n=== Cross-layer Check: Gold props match Silver ===")
    try:
        gold_props  = spark.table("gold_property_summary").select("PropertyKey").distinct().count()
        silver_props = spark.table("dimproperty__0001").filter(F.col("IsCurrent")==1).select("PropertyKey").distinct().count()
        match = gold_props == silver_props
        print(f"  {'OK' if match else 'WARN'} Gold properties: {gold_props} | Silver: {silver_props}")
    except Exception as e:
        print(f"  WARN cross-check: {e}")

    failed = [c for c in checks if not c["ok"]]
    print(f"\nGold Validation: {len(checks)-len(failed)}/{len(checks)} tables OK")
    if failed:
        print(f"  WARN: {[c['tbl'] for c in failed]} have 0 rows (check Silver layer ran first)")

## Pipeline Runner

In [ ]:
# ============================================================
# PIPELINE RUNNER — Gold
# ============================================================
import traceback

_RESULTS = []

def _run(name, fn):
    print(f"\n{'='*55}")
    print(f"START: {name}")
    print("="*55)
    try:
        fn()
        _RESULTS.append((name, "OK"))
    except Exception as e:
        _RESULTS.append((name, str(e)[:300]))
        print(f"FAIL: {name}: {e}")
        traceback.print_exc()

_run("Gold 1 — Property Summary",    build_gold_property_summary)
_run("Gold 2 — AR Summary",          build_gold_ar_summary)
_run("Gold 3 — Maintenance Summary", build_gold_maintenance_summary)
_run("Gold 4 — Leasing Summary",     build_gold_leasing_summary)
_run("Gold 5 — Financial Summary",   build_gold_financial_summary)
_run("Gold 6 — Validation",          validate_gold)

print("\n" + "="*55)
print("GOLD PIPELINE SUMMARY")
print("="*55)
for name, status in _RESULTS:
    icon = "OK" if status == "OK" else "FAIL"
    print(f"  [{icon}] {name}" + (f": {status}" if status != "OK" else ""))

if any(s != "OK" for _,s in _RESULTS):
    raise Exception("Gold pipeline had failures.")
print("\nGold layer complete. Ready for Semantic Model.")